# CIFAR-10 - CNN Model Comparison

Notebook này chạy 4 kiến trúc CNN trên một dataset ảnh để so sánh hiệu năng.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "dataset"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")
tf.random.set_seed(42)
np.random.seed(42)


In [ ]:
data = np.load(DATA_DIR / "cifar10.npz")
x_train, y_train = data["x_train"], data["y_train"].reshape(-1)
x_test, y_test = data["x_test"], data["y_test"].reshape(-1)
x_train = x_train / 255.0
x_test = x_test / 255.0
input_shape = x_train.shape[1:]


In [ ]:
def build_simple_cnn(input_shape, num_classes):
    return keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])

def build_lenet(input_shape, num_classes):
    return keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(6, 5, activation='tanh'),
        layers.AveragePooling2D(),
        layers.Conv2D(16, 5, activation='tanh'),
        layers.AveragePooling2D(),
        layers.Flatten(),
        layers.Dense(120, activation='tanh'),
        layers.Dense(84, activation='tanh'),
        layers.Dense(num_classes, activation='softmax')
    ])

def residual_block(x, filters):
    shortcut = x
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, 3, padding='same')(x)
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, padding='same')(shortcut)
    x = layers.Add()([x, shortcut])
    return layers.Activation('relu')(x)

def build_resnet_lite(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(inputs)
    x = residual_block(x, 32)
    x = layers.MaxPooling2D()(x)
    x = residual_block(x, 64)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inputs, outputs)

def build_vgg_small(input_shape, num_classes):
    return keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])

builders = {
    "SimpleCNN": build_simple_cnn,
    "LeNet5": build_lenet,
    "ResNetLite": build_resnet_lite,
    "VGGSmall": build_vgg_small,
}

results = []
histories = {}
for name, builder in builders.items():
    model = builder(input_shape, 10)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    history = model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=3, batch_size=128, verbose=0)
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    results.append({"model": name, "test_accuracy": acc, "test_loss": loss})
    histories[name] = history.history

results_df = pd.DataFrame(results).sort_values("test_accuracy", ascending=False)
results_df

ax = sns.barplot(data=results_df, x="test_accuracy", y="model", palette="viridis")
ax.set_title("Model Comparison - CIFAR-10")
plt.show()
